<a id="unit-u.94e060b81a38472a8f8fb09899ce3a2f"></a>

<div>
  <div>7-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Ретсіз оқиғаларды шешу, тарихты сақтау және қалпына келетіндей қайта ойнату</div>
</div>

Тапсырыс оқиғалары екі рет, ретсіз және үзілістен кейін келеді. Әр тапсырыстың ағымдағы күйін бизнес нұсқаларымен сақтайсыз, әр бөлек оқиғаны тарихта бір рет сақтайсыз, әр жеткізілімді тіркейсіз және модельденген сәтсіздіктен кейін үшеуі де төлемдер, қайтарымдар және жөнелтулердің тәуелсіз тізіліміне сәйкес келгенше бүкіл пакетті қайта ойнатасыз. Содан кейін бөлек көшірмелерде тапсырыстың бір бағанын жаңартып, тапсырыстарды жоясыз. Шамамен 45 минут жоспарлаңыз.

**7-модуль: деректерді жаңарту, жою және оқиғаларды қайта ойнату** · [Сабақ](course.md) · [Тест](quiz07_state_changes_and_replay.ipynb) · [Курстың README файлы](../../README.md)


<a id="unit-u.9c141344418d4cf0803c439ed6d9c864"></a>

<a id="prerequisites-and-business-sources"></a>

## Алғышарттар және бизнес дереккөздері

Алдымен 6-модульді аяқтаңыз. Бұл жұмыс orders_clean ішінен қабылданған тапсырыстарды оқып, COURSE_SIMULATION дереккөзімен 900001–900011 тапсырыстарының модельденген өзгерістерін өңдейді.

Тек orders_update_walkthrough, orders_replay_practice, orders_current, order_events, event_deliveries, orders_partial_update, orders_delete_demo, order_items, products, payments, refunds және shipment_events кестелерін қайта жасаңыз.

Қалыпты ағым: жасалды → төленді → жөнелтілді → жеткізілді; қайтарым ағымы: жасалды → төленді → бас тартылды → қайтарылды.
Бұл жұмыс файлдан модельденген оқиғаларды оқып, жеткізілген оқиғаны төленген оқиғадан бұрын жеткізеді. Әр оқиға толық жаңартылған күйді (after-image) қамтиды; бір оқиғаның қайталанған жеткізілімдері бірдей мазмұнда.

wwi_products жоқ болса, бизнес тізілімін баптау WWI Parquet пакетінен 227 жолын жүктейді. Бар wwi_products кестесі өзгеріссіз қайта пайдаланылады.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response
from dw_course.wwi import manifest, parquet_ddl, parquet_paths

lab = connect_sandbox(module=7)




<a id="unit-u.14296151b42d4b72b7f2d38bc244c52a"></a>

<a id="start-with-a-small-example-normal-update-duplicate-and-out-of-order-events"></a>

## Шағын мысалдан бастау: қалыпты жаңарту, қайталану және ретсіз оқиғалар

Бұл қадам 6-модульдің қабылданған кестесін сақтап, тек orders_update_walkthrough кестесін қайта жасайды. Алдымен тапсырыстың CREATED күйінен PAID күйіне өзгеруін бақылаңыз, сол оқиғаны қайталаңыз, содан кейін SHIPPED және кешіккен PAID оқиғасын алыңыз. Әр қадамнан кейін ағымдағы күйді сұраңыз.

Алдымен көрсетілген кесте жасау SQL-ін қарап, үш параметрді табыңыз:

| Параметр | Мақсаты |
| --- | --- |
| UNIQUE KEY(order_id) | Бір тапсырысты анықтайды |
| "enable_unique_key_merge_on_write"="true" | Merge-on-Write моделін қолданады |
| "function_column.sequence_col"="event_version" | Бір тапсырыстың бизнес нұсқаларын салыстыратын өрісті көрсетеді |

order_ddl(..., current=True) — дерекқор синтаксисі емес, осы DDL жасайтын курс көмекшісі.
Жай event_version атаулы баған қосу нұсқаларды шешуді қоспайды; оны Sequence бағаны ретінде баптау қажет.
Әр келесі оқиға толық жол мен нұсқа береді. 3-нұсқа 2-нұсқадан бұрын келсе, 3-нұсқа сақталады.
Бірдей нұсқаның басқа мазмұны бұл жұмыста қауіпсіз қайта жеткізу емес; бөлек қайшылық ережелері қажет.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_update_walkthrough")
ddl = order_ddl("orders_update_walkthrough", current=True)
show_sql("Inspect the primary key and Sequence configuration first", ddl)
lab.execute(ddl)
initial = dict(fixture("orders.json")[0])
lab.insert("orders_update_walkthrough", ORDER_COLUMNS, order_rows([initial]))
lab.sql("SELECT order_id,status,event_version FROM orders_update_walkthrough", title="Initial order");

<a id="unit-u.6c38d348f52b41eb8082b12bcf49c97c"></a>

**Күтілетін нәтиже**

`UNIQUE KEY(order_id)`, `"enable_unique_key_merge_on_write" = "true"` және `"function_column.sequence_col" = "event_version"` бар кесте анықтамасы, содан кейін бір жол: 900001 тапсырысы, CREATED, 1-нұсқа.


<a id="unit-u.5113a35b71184f029ce6d36a56cacd6c"></a>

<a id="normal-update"></a>

### Қалыпты жаңарту

2-нұсқа аяқталған төлемді білдіреді. Жаңа оқиға тапсырыс ID-ін өзгертпей, толық тапсырыс күйін қамтиды.


In [ ]:
paid = dict(initial, status="PAID", event_version=2, event_id="WALK_PAID", paid_amount="100.00")
lab.insert("orders_update_walkthrough", ORDER_COLUMNS, order_rows([paid]))
lab.sql("SELECT order_id,status,event_version,paid_amount FROM orders_update_walkthrough", title="Current state after payment")
expect(lab.query("SELECT status,event_version FROM orders_update_walkthrough"), [("PAID",2)], title="PAID, version 2")

<a id="unit-u.532264342af2418795576b1e18bd3fe3"></a>

**Күтілетін нәтиже**

900001 үшін әлі бір жол, енді PAID, 2-нұсқа, 100.00 төленген, жасыл белгі. Жаңа жол ескісін ауыстырды, себебі кілттері бірдей, жаңа нұсқа жоғарырақ.


<a id="unit-u.6c32142875074a4aaf240734d7d6726f"></a>

<a id="duplicate-delivery"></a>

### Қайталанған жеткізілім

Сол төлем оқиғасын қайта жазыңыз. Тапсырысқа тағы бір жол қосыла ма, әлде бір жол болып қала ма, болжаңыз, содан кейін нәтижені қараңыз.


In [ ]:
lab.insert("orders_update_walkthrough", ORDER_COLUMNS, order_rows([paid]))
lab.sql("SELECT order_id,status,event_version,paid_amount FROM orders_update_walkthrough", title="The same event arrives again")
expect(lab.query("SELECT COUNT(*),MAX(event_version) FROM orders_update_walkthrough"), [(1,2)], title="Still one row, version 2")

<a id="unit-u.9dc9d98f395441349c2b61dbc0e8ff1e"></a>

**Күтілетін нәтиже**

Сол бір жол, PAID, 2-нұсқа, жасыл белгі: бір жол, ең жоғары нұсқа — 2. Сол оқиғаның екінші рет келуі ештеңені өзгертпейді.


<a id="unit-u.081bc6445b2c46c4bb761c04f515616d"></a>

<a id="out-of-order-delivery"></a>

### Ретсіз жеткізілім

Алдымен 3-нұсқаның жөнелтуін өңдеп, содан кейін 2-нұсқаның ескі төлемін қайта жеткізіңіз. Sequence бағаны ағымдағы күйді бизнес нұсқасы бойынша таңдайды.


In [ ]:
shipped = dict(paid, status="SHIPPED", event_version=3, event_id="WALK_SHIPPED")
lab.insert("orders_update_walkthrough", ORDER_COLUMNS, order_rows([shipped]))
lab.insert("orders_update_walkthrough", ORDER_COLUMNS, order_rows([paid]))
lab.sql("SELECT order_id,status,event_version,paid_amount FROM orders_update_walkthrough", title="The late payment event did not overwrite the shipped state")
expect(lab.query("SELECT status,event_version FROM orders_update_walkthrough"), [("SHIPPED",3)], title="Still SHIPPED, version 3")

<a id="unit-u.d135ab7ba25b4b6f9c07eab47fb86f28"></a>

**Күтілетін нәтиже**

Бір жол, SHIPPED, 3-нұсқа, жасыл белгі. Кешіккен PAID 2-нұсқасы 3-нұсқадан кейін келіп, оны ауыстырмады: Sequence бағаны келу ретін емес, нұсқаларды салыстырады.


<a id="unit-u.8f12db927b7d482aacb6984a7edd53cf"></a>

<a id="1-create-separate-current-state-and-history-tables"></a>

## 1. Ағымдағы күй мен тарихқа бөлек кестелер жасау

Ағымдағы кесте UNIQUE KEY(order_id) қолданып, event_version бойынша жаңарақ күйді сақтайды; тарих кестесі UNIQUE KEY(event_id) қолданып, бөлек оқиғаларды сақтайды; жеткізілім кестесі әр әрекетті тіркейді.

Бұл кестелер бөлек қадамдарда жазылады. Келесіде бір кесте жаңартылып, екіншісі жаңартылмағандағы қалпына келтіруді бақылаңыз.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_current")
ddl = order_ddl("orders_current", current=True)
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS order_events")
ddl = order_ddl("order_events", history=True)
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
lab.execute(f"INSERT INTO orders_current ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_clean")
lab.execute(f"INSERT INTO order_events ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_clean")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_current"), [(10,"1400.00")],
       title="Ten current orders from Module 6, 1400.00")
lab.execute("DROP TABLE IF EXISTS event_deliveries")
lab.execute('CREATE TABLE event_deliveries (attempt_id BIGINT, delivery_id BIGINT, event_id VARCHAR(32), payload STRING) DUPLICATE KEY(attempt_id, delivery_id) DISTRIBUTED BY HASH(attempt_id) BUCKETS 1 PROPERTIES("replication_num"="1")');

<a id="unit-u.20e8dd5dbf9f4eff923c4d6191dd5a22"></a>

**Күтілетін нәтиже**

Екі кесте анықтамасы: `event_version` Sequence бағаны және `order_id` кілті бар `orders_current`, `event_id` кілті бар `order_events`. Жасыл белгі: 6-модульдің он қабылданған тапсырысы, 1400.00, ағымдағы күй және алғашқы тарих болады.


<a id="unit-u.e8cab60335ad48feace0ded3caf97cae"></a>

<a id="2-simulate-an-interruption-then-recover"></a>

## 2. Үзілісті модельдеу, содан кейін қалпына келтіру

Алғашқы жеткізілімді тіркеп, тарихты жазыңыз, содан кейін ағымдағы кестені жазбас бұрын үзілісті модельдеңіз. Әр жеткізілімді сақтап, тұрақты оқиға ID-і арқылы тарих қайталануларын жоя отырып, бүкіл оқиғалар пакетін қайта жеткізіңіз. Бұл тәжірибе дерекқор процесін емес, тек курс қадамдарын үзеді.


In [ ]:
import json
deliveries = fixture("deliveries.json")
first = deliveries[0]
lab.insert("event_deliveries", ["attempt_id","delivery_id","event_id","payload"],
           [(0, first["delivery_id"], first["event_id"], json.dumps(first))])
lab.insert("order_events", ORDER_COLUMNS, order_rows([first]))
expect(lab.query("SELECT status FROM orders_current WHERE order_id=900001"), [("CREATED",)],
       title="Interrupted: the history has the first event, the current table does not")

def replay(attempt):
    for record in deliveries:
        lab.insert("event_deliveries", ["attempt_id","delivery_id","event_id","payload"],
                   [(attempt, record["delivery_id"], record["event_id"], json.dumps(record))])
        lab.insert("order_events", ORDER_COLUMNS, order_rows([record]))
        lab.insert("orders_current", ORDER_COLUMNS, order_rows([record]))

replay(1)

<a id="unit-u.e84aa78e5108470a9b874c392bac55a8"></a>

**Күтілетін нәтиже**

Жасыл белгі: үзілістен кейін 900001 тапсырысы `orders_current` ішінде әлі CREATED, бірақ алғашқы жеткізілген оқиғасы тарихта бар. Содан кейін алғашқы толық қайта ойнату ештеңе шығармай орындалады.


<a id="unit-u.ccfb89a9e8134a169a23556d6c78e44c"></a>

<a id="3-reconcile-row-by-row-then-replay-the-entire-batch"></a>

## 3. Әр жолды салыстыру, содан кейін бүкіл пакетті қайта ойнату

900001 DELIVERED, 4-нұсқа болып қалады; 900003 REFUNDED, 4-нұсқа болып, жинақталған төлемі әлі 150.00 қалады.
10 бастапқы snapshot және 8 бөлек оқиға 18 логикалық тарих жазбасын және 11 ағымдағы тапсырысты береді.
Әр пакетте бір қайталануды қоса 9 жеткізілім бар; үзіліске дейінгі 1 жеткізілім және бүкіл пакетті екі рет қайта ойнату 19 бастапқы жеткізілім береді.


In [ ]:
projection = ",".join("DATE_FORMAT(event_time, '%Y-%m-%d %H:%i:%s')" if col == "event_time" else col for col in ORDER_COLUMNS)
def verify_state():
    expect(lab.query(f"SELECT {projection} FROM orders_current ORDER BY order_id"), order_rows(fixture("expected_current.json")))
    expect(lab.query("SELECT COUNT(*), SUM(order_amount), SUM(paid_amount), SUM(refund_amount) FROM orders_current"),
           [(11,"1510.00","250.00","150.00")])
    expect(lab.query("SELECT COUNT(*) FROM order_events"), [(18,)])
    expected_history = {r["event_id"]:r for r in fixture("orders.json") + deliveries}
    expect(lab.query(f"SELECT {projection} FROM order_events ORDER BY event_id"),
           order_rows([expected_history[key] for key in sorted(expected_history)]),
           title="Current state and history match the expected orders and events")
verify_state()
replay(2)
verify_state()
expect(lab.query("SELECT COUNT(*) FROM event_deliveries"), [(19,)], title="Nineteen raw deliveries")
lab.sql("SELECT 'Current orders' AS object,COUNT(*) AS records FROM orders_current UNION ALL SELECT 'Business history',COUNT(*) FROM order_events UNION ALL SELECT 'Raw deliveries',COUNT(*) FROM event_deliveries ORDER BY records", title="Counts for each of the three record types");

<a id="unit-u.f96eb48b540b4037b0e6f710fde27d47"></a>

**Күтілетін нәтиже**

Үш жасыл белгі: 1-қайта ойнатудан кейінгі алғашқы тексеру, 2-қайта ойнатудан кейінгі екіншісі және 19 бастапқы жеткізілім. Содан кейін үш сан: 11 ағымдағы тапсырыс, 18 тарих жазбасы, 19 бастапқы жеткізілім. Бүкіл пакетті екінші рет қайта ойнату тоғыз жеткізілім қосып, ағымдағы күйді де, тарихты да өзгертпеді.


<a id="unit-u.1b4f37c70bb54efd94408e140117ac02"></a>

<a id="independently-verify-current-state-with-business-transaction-records"></a>

### Ағымдағы күйді бизнес транзакцияларының жазбаларымен тәуелсіз тексеру

business_events.json ішінде бөлек тізілген тауарлармен, төлемдермен, қайтарымдармен және жөнелту оқиғаларымен салыстырыңыз: тауарлар сомасы тапсырыс сомасына, төлемдер қосындысы 250.00-ге, қайтарымдар 150.00-ге тең болуы керек. Төлемдер мен қайтарымдардың қайталанулары өз бизнес ID-лері арқылы жойылады.

Қайтарымдар бастапқы төлемдерге, жөнелту оқиғалары тапсырыстарға байланысуы керек. Өнім өлшемі 5-модульдегі wwi_products кестесінен алынады (227 жол). Сол бизнес транзакцияларын қайта жазғаннан кейін бөлек бизнес жазбаларының саны мен сомалары өзгермеуі керек.

wwi_products жоқ болса, бизнес тізілімін баптау WWI Parquet пакетінен 227 жолын жүктейді. Бар wwi_products кестесі өзгеріссіз қайта пайдаланылады.


In [ ]:
business = fixture("business_events.json")
definitions = {
    "order_items": 'order_line_id BIGINT NOT NULL, order_id BIGINT, product_id BIGINT, quantity INT, unit_price DECIMAL(12,2)',
    "products": 'product_id BIGINT NOT NULL, product_name STRING',
    "payments": 'payment_id VARCHAR(32) NOT NULL, order_id BIGINT, event_id VARCHAR(32), amount DECIMAL(12,2), event_time DATETIME',
    "refunds": 'refund_id VARCHAR(32) NOT NULL, payment_id VARCHAR(32), order_id BIGINT, event_id VARCHAR(32), amount DECIMAL(12,2), event_time DATETIME',
    "shipment_events": 'shipment_event_id VARCHAR(32) NOT NULL, shipment_id VARCHAR(32), order_id BIGINT, event_id VARCHAR(32), status VARCHAR(20), event_time DATETIME',
}
for table, fields in definitions.items():
    key = fields.split()[0]
    lab.execute("DROP TABLE IF EXISTS " + table)
    ddl = f'CREATE TABLE {table} ({fields}) UNIQUE KEY({key}) DISTRIBUTED BY HASH({key}) BUCKETS 1 PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")'
    show_sql("Business transaction tables", ddl)
    lab.execute(ddl)
if not lab.query("SHOW TABLES LIKE 'wwi_products'"):
    paths = parquet_paths()
    metadata = manifest()["tables"]["products"]
    lab.execute(parquet_ddl("products", "wwi_products"))
    response = lab.stream_load(
        "wwi_products", paths["products"], "module7_wwi_" + uuid4().hex, format="parquet"
    )
    show_response(response)
    expect(response["Status"], "Success")
    expect(response["NumberLoadedRows"], metadata["rows"])
    expect(response["NumberFilteredRows"], 0)

lab.execute("INSERT INTO products SELECT StockItemID, StockItemName FROM wwi_products")
expect(lab.query("SELECT COUNT(*) FROM products"), [(227,)], title="The product dimension has 227 products")
for attempt in range(2):
    for table, records in [("order_items", business["order_lines"]), ("payments", business["payments"]),
                           ("refunds", business["refunds"]), ("shipment_events", business["shipments"])]:
        columns = list(records[0])
        lab.insert(table, columns, [tuple(r[c] for c in columns) for r in records])
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM payments"), [(2, "250.00")], title="Two payments, 250.00, after loading the ledger twice")
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM refunds"), [(1, "150.00")], title="One refund, 150.00")
expect(lab.query("SELECT COUNT(*) FROM shipment_events"), [(2,)], title="Two shipment events")
expect(lab.query("SELECT COUNT(*) FROM order_items i LEFT JOIN products p ON i.product_id=p.product_id WHERE p.product_id IS NULL"), [(0,)])
expect(lab.query("SELECT COUNT(*) FROM orders_current o LEFT JOIN customers c ON o.customer_id=c.customer_id WHERE c.customer_id IS NULL"), [(0,)])
expect(lab.query("""
SELECT COUNT(*) FROM refunds r LEFT JOIN payments p
ON r.payment_id=p.payment_id AND r.order_id=p.order_id
WHERE p.payment_id IS NULL OR r.amount>p.amount
"""), [(0,)])
for table in ("payments", "refunds", "shipment_events"):
    expect(lab.query(f"SELECT COUNT(*) FROM {table} b LEFT JOIN order_events h ON b.event_id=h.event_id AND b.order_id=h.order_id WHERE h.event_id IS NULL"), [(0,)])
expect(lab.query("""
SELECT COUNT(*) FROM orders_current o
LEFT JOIN (SELECT order_id, SUM(quantity*unit_price) amount FROM order_items GROUP BY order_id) i ON o.order_id=i.order_id
LEFT JOIN (SELECT order_id, SUM(amount) paid FROM payments GROUP BY order_id) p ON o.order_id=p.order_id
LEFT JOIN (SELECT order_id, SUM(amount) refunded FROM refunds GROUP BY order_id) r ON o.order_id=r.order_id
WHERE i.order_id IS NULL OR o.order_amount<>i.amount
   OR o.paid_amount<>COALESCE(p.paid,0) OR o.refund_amount<>COALESCE(r.refunded,0)
"""), [(0,)], title="Every current order agrees with its items, payments, and refunds")
lab.sql("SELECT order_id, status, event_version, paid_amount, refund_amount, data_source FROM orders_current ORDER BY order_id", title="Current order state and source");
lab.sql("""SELECT COUNT(*) AS current_orders, SUM(order_amount) AS ordered_amount,
       SUM(paid_amount) AS paid_amount, SUM(refund_amount) AS refunded_amount,
       SUM(paid_amount-refund_amount) AS net_collections
FROM orders_current""", title="Current simulated orders and cash: one completion summary");


<a id="unit-u.6f49c0dc99864120b5a4dffeea9c7be5"></a>

**Күтілетін нәтиже**

Бес кесте анықтамасы, 227 өнімге, жалпы 250.00 болатын екі төлемге, 150.00 бір қайтарымға, екі жөнелту оқиғасына және әр ағымдағы тапсырыстың тауарларына, төлемдері мен қайтарымдарына сәйкес екеніне жасыл белгілер; тізілім екі рет жазылса да, әр жазбаны бір рет санайды. `wwi_products` жоқ болса, алдымен 227 жолдық Stream Load жауабы шығады.

Ағымдағы тапсырыстар: 900001 — DELIVERED, 4-нұсқа, 100.00 төленген; 900002 — CANCELLED, 2-нұсқа; 900003 — REFUNDED, 4-нұсқа, 150.00 төленген және 150.00 қайтарылған; 900004–900010 — CREATED, 1-нұсқа; 900011 — жаңа CREATED тапсырысы. Бәрі `COURSE_SIMULATION`.
Аяқтау қорытындысында бір жол бар: **11 ағымдағы тапсырыс, тапсырыс сомасы 1510.00, төленгені 250.00, қайтарылғаны 150.00, таза түсім 100.00**. Бұлар WWI шот түсімдері емес, модельденген ағымдағы тапсырыстар. Қайтарылған тапсырыс төлемін де, қайтарымын да сақтайды; таза түсімді табу үшін айырмасын алыңыз.


<a id="unit-u.61cb8f468ac34e429ce1e72530139b02"></a>

<a id="4-partial-column-updates"></a>

## 4. Бағандарды ішінара жаңарту

Тәуелсіз orders_partial_update кестесінде бағандарды ішінара жаңартуды уақытша қосып, тапсырыс ID-ін, жаңа күйді және өсетін нұсқаны жіберіңіз. Күй CANCELLED болғанын, ал сома 100.00, аймақ EAST болып қалғанын тексеріңіз. Кейін сеанс параметрлерін қалпына келтіріңіз.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_partial_update")
ddl = order_ddl("orders_partial_update", current=True)
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
lab.insert("orders_partial_update", ORDER_COLUMNS, order_rows(fixture("orders.json")[:1]))
previous_partial = lab.query("SELECT @@enable_unique_key_partial_update")[0][0]
try:
    lab.execute("SET enable_unique_key_partial_update = true")
    lab.execute("INSERT INTO orders_partial_update (order_id, status, event_version) VALUES (900001, 'CANCELLED', 2)")
finally:
    lab.execute("SET enable_unique_key_partial_update = %s", (previous_partial,))
expect(lab.query("SELECT status, event_version, order_amount, region FROM orders_partial_update"),
       [("CANCELLED",2,"100.00","EAST")], title="Status and version changed; amount and region kept")

<a id="unit-u.c6917b32663549bc8e518fb8bf4b7977"></a>

**Күтілетін нәтиже**

Кесте анықтамасы және жасыл белгі: 900001 тапсырысы CANCELLED, 2-нұсқа, сома 100.00 және аймақ EAST өзгермеген, `INSERT` тек үш бағанды атаған болса да. Сеанс параметрі бұрынғы мәніне қайтты.


<a id="unit-u.caefdb22bce24f6da90622006582d332"></a>

<a id="5-independent-deletion-experiment"></a>

## 5. Тәуелсіз жою тәжірибесі

Негізгі ағымда қайтарылған тапсырыстарды жоймаңыз. Көшірмеде бизнес жұмсақ жоюы үшін is_deleted, содан кейін басқа жол үшін SQL DELETE қолданыңыз; сұрауға көрінбеу дисктің бірден босауын білдірмейді. `UPDATE` тек осы көшірме сияқты Unique Key кестелерінде істейді; Duplicate не Aggregate Key кестесінде Doris `Only unique table could be updated` қатесімен бас тартады.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_delete_demo")
lab.execute('CREATE TABLE orders_delete_demo (order_id BIGINT, is_deleted BOOLEAN, status VARCHAR(20)) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")')
lab.insert("orders_delete_demo", ["order_id","is_deleted","status"], [(900001,False,"CREATED"),(900002,False,"CREATED")])
lab.execute("UPDATE orders_delete_demo SET is_deleted=true WHERE order_id=900001")
expect(lab.query("SELECT COUNT(*) FROM orders_delete_demo"), [(2,)], title="Soft deletion keeps both rows")
expect(lab.query("SELECT order_id FROM orders_delete_demo WHERE is_deleted=false"), [(900002,)], title="Only a filter hides the soft-deleted order")
lab.execute("DELETE FROM orders_delete_demo WHERE order_id=900002")
expect(lab.query("SELECT order_id FROM orders_delete_demo"), [(900001,)], title="DELETE removed 900002 from every query")
verify_state()

<a id="unit-u.8f203786e35b4a6ea8c39cf1fbe35955"></a>

**Күтілетін нәтиже**

Төрт жасыл белгі: жұмсақ жоюдан кейін көшірмеде әлі 2 жол бар, 900001-ді тек `WHERE is_deleted=false` жасырады; `DELETE` кейін кәдімгі сұраулар тек 900001 қайтарады. Соңғы тексеру 3-қадамды қайталайды, сондықтан негізгі кестелер өзгермеген.


<a id="unit-u.022325dd710c4cb1b3e2d08d86942a0c"></a>

<a id="6-practice-the-history-of-a-refunded-order"></a>

## 6. Жаттығу: қайтарылған тапсырыстың тарихы

`order_events` ішінен 900003 тапсырысының тарихын `event_time` ретімен, күйімен, нұсқасымен, төленген және қайтарылған сомасымен, сондай-ақ тізілімдегі төлемі мен қайтарымын көрсетіңіз. Содан кейін түсіндіріңіз: қайтарымнан кейін `paid_amount` неге 150.00 қалады және таза алынған соманы қайдан оқисыз?


In [ ]:
# Your code here

<a id="unit-u.5e189b3d762a4244b7c8e15bb6997bce"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
lab.sql("""SELECT event_id, event_version, status, event_time, paid_amount, refund_amount
FROM order_events WHERE order_id = 900003 ORDER BY event_time, event_version""", title="The history of order 900003");
lab.sql("""SELECT p.payment_id, p.amount AS paid, r.refund_id, r.amount AS refunded
FROM payments p LEFT JOIN refunds r ON r.payment_id = p.payment_id
WHERE p.order_id = 900003""", title="Its payment and refund in the ledger");
expect(lab.query("SELECT paid_amount, refund_amount, paid_amount - refund_amount FROM orders_current WHERE order_id = 900003"),
       [("150.00", "150.00", "0.00")], title="Paid 150.00, refunded 150.00, nothing received net")

<a id="unit-u.0b57f819d7d44d4cb99ae5f7091ae238"></a>

**Күтілетін нәтиже**

Төрт оқиға: `S1003` CREATED, 1-нұсқа; `E04` PAID, 2-нұсқа, 150.00 төленген; `E07` CANCELLED, 3-нұсқа; `E05` REFUNDED, 4-нұсқа, 150.00 қайтарылған. Тізілімде 150.00 сомалы `P03` төлемі және соған қатысты 150.00 сомалы `R03` қайтарымы бар, тексеру жасыл.

`paid_amount` төленгенді тіркейді, қайтарым төлем фактісін жоймайды: бұл `refund_amount` ретінде тіркелген екінші факт. Таза түсім — `paid_amount - refund_amount`, мұнда 0.00. Нөл көрсету үшін төлемді жою ақшаның келіп, қайта кеткен фактісін өшірер еді.


<a id="unit-u.10f6d36840a74039bc9c9f17e0d5aefb"></a>

<a id="7-practice-replay-a-late-event-twice"></a>

## 7. Жаттығу: кешіккен оқиғаны екі рет қайта ойнату

900001 тапсырысының ағымдағы күйін тәуелсіз orders_replay_practice кестесіне көшіріңіз, содан кейін PAID күйі және 2-нұсқасы бар толық ескі оқиға жасап, қатарынан екі рет жеткізіңіз. Орындап, тексермес бұрын күйді, нұсқаны және жолдар санын болжаңыз. Әлі DELIVERED, 4-нұсқадағы бір тапсырыс күтіледі.

Бұл жаттығу тек orders_replay_practice кестесін қайта жасайды; негізгі тарих пен жеткізілім кестелері өзгермейді.

Кодты келесі ұяшыққа жазыңыз, содан кейін одан кейінгі үлгі шешіммен салыстырыңыз.


In [ ]:
# Your code here

<a id="unit-u.7774355a1e8e4b79ae84f9e75d2fe581"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_replay_practice")
ddl = order_ddl("orders_replay_practice", current=True)
show_sql("Independent replay exercise table", ddl)
lab.execute(ddl)
lab.execute("INSERT INTO orders_replay_practice SELECT * FROM orders_current WHERE order_id=900001")
late_event = """INSERT INTO orders_replay_practice
(order_id,customer_id,order_amount,status,event_version,event_id,event_time,paid_amount,refund_amount,region,data_source)
SELECT order_id,customer_id,order_amount,'PAID',2,'PRACTICE_OLD','2026-01-01 10:00:00',
paid_amount,refund_amount,region,data_source FROM orders_current WHERE order_id=900001"""
lab.execute(late_event)
lab.execute(late_event)
lab.sql("SELECT order_id,status,event_version,paid_amount FROM orders_replay_practice", title="Result after redelivering the late old event twice");
expect(lab.query("SELECT order_id,status,event_version,paid_amount FROM orders_replay_practice"), [(900001,"DELIVERED",4,"100.00")],
       title="Still one order, DELIVERED at version 4")

<a id="unit-u.fce6b5be1bb449ceb22d1a16cded917c"></a>

**Күтілетін нәтиже**

Кесте анықтамасы, содан кейін бір жол: 900001 тапсырысы, DELIVERED, 4-нұсқа, 100.00 төленген, жасыл белгі. Ескі 2-нұсқа оқиғасының екі жеткізілімі ештеңені өзгертпеді: Sequence жоғары нұсқаны, Unique Key бір жолды сақтайды.


<a id="unit-u.ca586c9c03114949b0825b94c62e44b3"></a>

<a id="you-are-done-when"></a>

## Жұмысты аяқтау шарттары

- `orders_current` ішінде 11 тапсырыс, `order_events` ішінде 18 оқиға, `event_deliveries` ішінде 19 жеткізілім бар, әр санды түсіндіре аласыз;
- қайталану мен кешіккен оқиға ағымдағы күйді өзгертпегенін көрдіңіз, мұны қай параметр қамтамасыз ететінін айта аласыз;
- ағымдағы күй тәуелсіз тізілімге сәйкес: тапсырыс сомасы 1510.00, төленгені 250.00, қайтарылғаны 150.00;
- бағандарды ішінара жаңартуды толық жол жазуынан, жұмсақ жоюды `DELETE` операторынан ажырата аласыз.


<a id="unit-u.4ad1245a31ea465fa4877fc1875dd16d"></a>

<a id="next"></a>

## Келесі қадам

[7-тест: деректерді жаңарту, жою және оқиғаларды қайта ойнату](quiz07_state_changes_and_replay.ipynb) файлын ашыңыз. Бұл — 1-деңгейдің соңғы зертханалық жұмысы; 2-деңгей [8-модуль: көріністер және материалданған көріністер](../../level2/module08-views-materialized-views/course.md) бөлімінен басталады.
